# NGSO SLS — Slice A (MVP): Coverage over India

Computes **satellites-in-view** and **coverage availability** for the Reliance-Jio 1600-sat dual shell (Primary 1200/40/1 @48°, Secondary 400/20/7 @70°, 650 km) over an India lat/lon grid, using an analytic Kepler+J2 propagator.

Runs in **Google Colab** (clones the repo) and in **local Jupyter Lab** (uses the installed package). Set `REPO_URL` below to your git remote for the Colab path.

In [ ]:
# === Setup: make ngso_sls importable (Colab + local Jupyter Lab) ===
# 1. Set REPO_URL to your git remote (the URL you `git push` to).
# 2. Private repo? Put a GitHub token in Colab 'Secrets' (key icon) named GITHUB_TOKEN,
#    or inline it in the URL. Public repo needs no token.
REPO_URL = "https://github.com/YOUR_ORG/spacetime-sls.git"  # <-- set this

import importlib.util, subprocess, sys, os, re

def _sh(cmd):
    print("$", cmd.replace(REPO_URL, REPO_URL))
    subprocess.run(cmd, shell=True, check=True)

if importlib.util.find_spec("ngso_sls") is None:   # skip on local Jupyter if installed
    url = REPO_URL
    try:  # optional private-repo auth via Colab secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
        if _tok and url.startswith("https://github.com/"):
            url = url.replace("https://", f"https://{_tok}@")
    except Exception:
        pass
    repo_dir = re.sub(r"\.git$", "", os.path.basename(REPO_URL.rstrip("/"))) or "repo"
    if not os.path.isdir(repo_dir):
        _sh(f"git clone {url} {repo_dir}")
    _sh(f"{sys.executable} -m pip install -q -e {repo_dir}")

import ngso_sls
print("ngso_sls", ngso_sls.__version__)

## Configure and run the coverage simulation

Knobs are simulation parameters: time window/step, AOR, grid resolution, and `k_coverage` (min satellites in view to count a cell as served).

> **Note (MVP):** this milestone runs a single in-memory pass (no spatial sharding / time-chunking yet — that arrives in Milestone 3). Memory scales with cells × satellites × timesteps, so keep the grid/step modest here (defaults below use a 2° grid at 60 s ≈ 0.6 GB). Finer grids and 24 h runs come with the M3 chunked engine.

In [ ]:
from datetime import datetime, timezone
from ngso_sls.presets import jio_constellation
from ngso_sls.config import TimeGrid, SimConfig
from ngso_sls.pipeline import run_coverage
from ngso_sls.grids.aor import INDIA_AOR

sim = SimConfig(
    jio_constellation(),
    TimeGrid(datetime(2026, 1, 1, tzinfo=timezone.utc), duration_s=3600.0, step_s=60.0),
    k_coverage=1,
)
res = run_coverage(sim, INDIA_AOR, grid_step_deg=2.0)
print(f"cells: {len(res['lat'])}  min_elev: {res['min_elev_deg']}°")
print(f"availability  mean={res['availability'].mean():.3f}  "
      f"min={res['availability'].min():.3f}  max={res['availability'].max():.3f}")

## Coverage map + CSV export

In [ ]:
from ngso_sls.viz.plots import plot_availability
from ngso_sls.io.csv_io import write_availability_csv

fig = plot_availability(res)
write_availability_csv(
    res, "coverage_availability.csv",
    manifest={"seed": sim.seed, "cell_layout": sim.cell_layout,
              "step_s": sim.time_grid.step_s, "propagator": "KeplerJ2"},
)
print("wrote coverage_availability.csv")
fig